In [1]:
# Variance Threshold Filtering (Python / Pandas / Scikit-Learn) 

import pandas as pd
from sklearn.feature_selection import VarianceThreshold

# Sample dataset with 4 features (Column C is constant, Column D has very low variance)

data = {
    'Feature_A': [12.5, 14.2, 10.1, 15.6, 13.8, 11.9],
    'Feature_B': [0.15, 0.45, 0.89, 0.22, 0.55, 0.33],
    'Feature_C': [5.0, 5.0, 5.0, 5.0, 5.0, 5.0],
    'Feature_D': [5.01, 5.0, 5.0, 5.0, 5.0, 5.0]
}

df = pd.DataFrame(data)

# Initialize VarianceThreshold with a threshold of 0.05

selector = VarianceThreshold(threshold=0.05)
selector.fit(df)

# Get boolean mask of selected features

selected_mask = selector.get_support()
filtered_df = df.loc[:, selected_mask]

print("Original shape:", df.shape)
print("Filtered shape:", filtered_df.shape)
print("Selected features:\n", filtered_df.head())

Original shape: (6, 4)
Filtered shape: (6, 2)
Selected features:
    Feature_A  Feature_B
0       12.5       0.15
1       14.2       0.45
2       10.1       0.89
3       15.6       0.22
4       13.8       0.55


In [5]:
# Correlation-Based Feature Selection with Pandas

import pandas as pd
import numpy as np

np.random.seed(42)
X = np.random.randn(100, 5)
y = X[:, 0] * 2.5 + X[:, 2] * (-1.2) + np.random.randn(100) * 0.1

df = pd.DataFrame(X, columns=['Feat_1', 'Feat_2', 'Feat_3', 'Feat_4', 'Feat_5'])
df['Target'] = y

# Calculate absolute correlation matrix with target

corr_matrix = df.corr().abs()
target_corr = corr_matrix['Target'].drop('Target')

# Filter features with correlation greater than 0.3

threshold = 0.3
selected_features = target_corr[target_corr > threshold].index.tolist()

print("Target correlations:\n", target_corr)
print("Selected features based on correlation > 0.3:", selected_features)

Target correlations:
 Feat_1    0.880597
Feat_2    0.200220
Feat_3    0.432436
Feat_4    0.089649
Feat_5    0.134157
Name: Target, dtype: float64
Selected features based on correlation > 0.3: ['Feat_1', 'Feat_3']


In [6]:
# Embedded Feature Selection using Lasso (L1 Regularization)

import numpy as np
from sklearn.linear_model import Lasso
from sklearn.datasets import make_regression

# Generate synthetic regression data with 20 features, where only 5 are informative

X, y = make_regression(n_samples=500, n_features=20, n_informative=5, random_state=42)

# Fit Lasso model with alpha penalty

lasso = Lasso(alpha=0.1)
lasso.fit(X, y)

# Identify non-zero coefficients

selected_indices = np.where(lasso.coef_ != 0)[0]
eliminated_indices = np.where(lasso.coef_ == 0)[0]

print("Lasso Coefficients:", np.round(lasso.coef_, 4))
print("Selected feature indices:", selected_indices)
print("Eliminated feature indices:", eliminated_indices)

Lasso Coefficients: [32.0311  0.      0.      0.      0.     10.1908  0.     -0.     -0.
  0.     -0.     -0.     -0.      0.      0.     58.1848  0.     -0.
  9.3089  7.0857]
Selected feature indices: [ 0  5 15 18 19]
Eliminated feature indices: [ 1  2  3  4  6  7  8  9 10 11 12 13 14 16 17]


In [7]:
# Recursive Feature Elimination (RFE) with Random Forest

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_wine

# Load dataset (13 features)

wine = load_wine()
X, y = wine.data, wine.target

# Initialize estimator and RFE

estimator = RandomForestClassifier(n_estimators=50, random_state=42)
selector = RFE(estimator, n_features_to_select=5, step=1)
selector = selector.fit(X, y)

# Output results

print("Feature Ranking (1 means selected):", selector.ranking_)
print("Selected mask boolean array:", selector.support_)

Feature Ranking (1 means selected): [1 6 8 5 4 3 1 9 7 1 2 1 1]
Selected mask boolean array: [ True False False False False False  True False False  True False  True
  True]


In [10]:
# Here is a complete, production-grade Scikit-Learn pipeline that integrates missing value imputation,
# feature scaling, Lasso-based feature selection (SelectFromModel), and rigorous cross-validation evaluation.

# Using Pipeline and SelectFromModel ensures that all preprocessing and
# feature selection steps are computed strictly within each cross-validation fold, completely eliminating Data Leakage.

import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

# 1. Generate synthetic regression dataset with 30 features

X, y = make_regression(n_samples=1000, n_features=30, n_informative=10, noise=0.5, random_state=42)

# Inject 5% missing values randomly to simulate real-world messy data

rng = np.random.RandomState(42)
mask = rng.binomial(1, 0.05, size=X.shape).astype(bool)
X[mask] = np.nan

# 2. Build the Production-Grade Pipeline
# Step 1: Impute missing values using the median of each feature
# Step 2: Scale features to zero mean and unit variance
# Step 3: Select features using Lasso (L1 regularization removes coefficients set to 0)
# Step 4: Train the final estimator (Random Forest Regressor) on the selected features

pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('feature_selector', SelectFromModel(Lasso(alpha=0.05, random_state=42))),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 3. Configure robust Cross-Validation

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 4. Evaluate the pipeline using cross_validate

scoring = ['neg_mean_squared_error', 'r2', 'neg_mean_absolute_error']
cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_estimator=True)

# 5. Aggregate and display results

print("--- Cross-Validation Performance ---")
print(f"R2 Scores per fold: {np.round(cv_results['test_r2'], 4)}")
print(f"Mean R2 Score:      {np.mean(cv_results['test_r2']):.4f} (+/- {np.std(cv_results['test_r2']):.4f})")
print(f"Mean RMSE:          {np.sqrt(-np.mean(cv_results['test_neg_mean_squared_error'])):.4f}")
print(f"Mean MAE:           {-np.mean(cv_results['test_neg_mean_absolute_error']):.4f}")

# Inspect features selected in the first fold's pipeline

first_fold_selector = cv_results['estimator'][0].named_steps['feature_selector']
selected_feature_indices = first_fold_selector.get_support(indices=True)
print(f"\nFeatures selected by Lasso in Fold 1: {selected_feature_indices}")

--- Cross-Validation Performance ---
R2 Scores per fold: [0.6716 0.719  0.672  0.6378 0.6849]
Mean R2 Score:      0.6771 (+/- 0.0262)
Mean RMSE:          102.5586
Mean MAE:           79.7435

Features selected by Lasso in Fold 1: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
